# Del dato climático al kWh

Cuaderno del taller, generado a partir de `presentacion.qmd`: las mismas celdas, en el mismo orden. Ejecuta las celdas de arriba hacia abajo.

6to Coloquio de Energía · IER-UNAM · Guillermo Barrios del Valle

In [ ]:
# Google Colab ya trae pandas, numpy y matplotlib; solo falta pvlib.
try:
    import pvlib
except ImportError:
    %pip install -q pvlib

# ¿Por qué Python?

## Ejemplo: irradiancia de cielo despejado en Torreón

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pvlib.location import Location

torreon = Location(latitude=25.54, longitude=-103.41,
                   tz="America/Mexico_City", altitude=1120, name="Torreón")

horas = pd.date_range("2026-03-21", "2026-03-22", freq="10min", tz=torreon.tz)
cielo = torreon.get_clearsky(horas, model="ineichen", linke_turbidity=3.0)

fig, ax = plt.subplots(figsize=(9, 3.8))
cielo[["ghi", "dni", "dhi"]].plot(ax=ax, lw=2)
ax.set_ylabel("Irradiancia (W/m²)")
ax.set_xlabel("Hora local")
ax.set_title("Equinoccio de primavera · cielo despejado")
ax.legend(["GHI", "DNI", "DHI"], frameon=False)
ax.grid(alpha=.3)
plt.tight_layout()

# El código, celda por celda

## Celda 1 · Las herramientas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pvlib.location import Location

Importamos las bibliotecas: pandas para tablas con tiempo, numpy para matemáticas, matplotlib para gráficas y pvlib para la física solar.

## Celda 2 · El lugar

In [ ]:
torreon = Location(latitude=25.54, longitude=-103.41,
                   tz="America/Mexico_City",
                   altitude=1120, name="Torreón")

Creamos el sitio: dónde está Torreón, en qué zona horaria y a qué altura.

## Celda 3 · El cuándo

In [ ]:
horas = pd.date_range("2026-03-21", "2026-03-22",
                      freq="10min", tz=torreon.tz)

Creamos el reloj: un instante cada 10 minutos durante el 21 de marzo, en hora local de Torreón.

## Celda 4 · El cielo despejado

In [ ]:
cielo = torreon.get_clearsky(horas, model="ineichen",
                             linke_turbidity=3.0)

Calculamos la irradiancia de un día sin nubes: una tabla con GHI, DNI y DHI para cada instante.

## Celda 5 · La gráfica

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.6))
cielo[["ghi", "dni", "dhi"]].plot(ax=ax, lw=2)
ax.set_ylabel("Irradiancia (W/m²)")
ax.set_xlabel("Hora local")
ax.legend(["GHI", "DNI", "DHI"], frameon=False)
ax.grid(alpha=.3)
plt.tight_layout()

## La tabla `cielo`: un renglón por instante, tres columnas

In [ ]:
dia = cielo.loc["2026-03-21 06:00":"2026-03-21 18:00"]
dia.iloc[::18].round(0)     # un renglón cada 3 horas

El índice es la hora local. Cada columna es una irradiancia en W/m².

## Columna `ghi` · global horizontal

In [ ]:
ax = cielo["ghi"].plot(figsize=(9, 2.6), lw=2)
ax.set_ylabel("GHI (W/m²)"); ax.set_ylim(0, 1100)

Todo lo que cae sobre un metro cuadrado de piso horizontal.

## Columna `dni` · directa normal

In [ ]:
ax = cielo["dni"].plot(figsize=(9, 2.6), lw=2)
ax.set_ylabel("DNI (W/m²)"); ax.set_ylim(0, 1100)

Lo que llega en línea recta desde el disco solar, medido de frente al Sol.

## Columna `dhi` · difusa horizontal

In [ ]:
ax = cielo["dhi"].plot(figsize=(9, 2.6), lw=2)
ax.set_ylabel("DHI (W/m²)"); ax.set_ylim(0, 1100)

Lo que llega del resto del cielo, sin el disco solar: la luz que hay en la sombra.

# Radiación sobre una superficie inclinada

## Celda 6 · La posición del Sol

In [ ]:
sol = torreon.get_solarposition(horas)
angulos = sol[["apparent_zenith", "azimuth"]]
angulos.loc[dia.index[::18]].round(1)

Para cada instante, dónde está el Sol: el ángulo cenital, desde la vertical, y el azimut, con 0° = N, 90° = E, 180° = S. La tabla `sol` trae más columnas; estas dos son las que usa la ecuación.

## Celda 7 · La ecuación en pvlib

In [ ]:
from pvlib import irradiance
pared = irradiance.get_total_irradiance(
    surface_tilt=90, surface_azimuth=90,
    solar_zenith=sol["apparent_zenith"],
    solar_azimuth=sol["azimuth"],
    dni=cielo["dni"], dhi=cielo["dhi"],
    ghi=cielo["ghi"], albedo=0.20,
    model="isotropic")

Una pared vertical (β = 90°) que mira al este (γ = 90°) con pasto enfrente (ρ = 0.20).

## La tabla `pared`: los tres términos y su suma

In [ ]:
pared.loc[dia.index[::18]].round(0)

`poa_direct`, `poa_sky_diffuse` y `poa_ground_diffuse` son los términos 1, 2 y 3. `poa_diffuse` suma los dos últimos y `poa_global` es el total: lo que recibe la pared.

# De la potencia a la energía

## Celda 8 · La energía del día

In [ ]:
dt_h = 10 / 60                        # horas por renglón
energia = pared["poa_global"].sum() * dt_h / 1000
print(f"{energia:.2f} kWh/m² en el día")

Sumamos todos los renglones de `poa_global` y multiplicamos por la duración de cada uno. Es el área bajo la curva, en kWh por metro cuadrado de pared.

## Celda 9 · Cuatro paredes

In [ ]:
orientaciones = {"Norte": 0, "Este": 90,
                 "Sur": 180, "Oeste": 270}
paredes = {}
for nombre, azimut in orientaciones.items():
    poa = irradiance.get_total_irradiance(
        surface_tilt=90, surface_azimuth=azimut,
        solar_zenith=sol["apparent_zenith"],
        solar_azimuth=sol["azimuth"],
        dni=cielo["dni"], dhi=cielo["dhi"],
        ghi=cielo["ghi"], albedo=0.20)
    paredes[nombre] = poa["poa_global"]
paredes = pd.DataFrame(paredes)

Un ciclo: la celda 7 cuatro veces, cambiando solo el azimut. Guardamos `poa_global` de cada pared en una tabla.

## Celda 10 · La energía de cada pared

In [ ]:
energias = paredes.sum() * dt_h / 1000
energias.round(2)

kWh/m² en el día para cada orientación, con el mismo cálculo de la celda 8 aplicado a las cuatro columnas a la vez.